# RobustScaler 

In this notebook, we will learn **RobustScaler** step by step.

### What we will learn
- What is RobustScaler?
- What are outliers?
- Median and IQR
- `fit()`, `transform()`, and `fit_transform()`
- Train/test scaling and data leakage
- RobustScaler vs StandardScaler vs MinMaxScaler


> **Simple idea:** RobustScaler is useful when data contains **outliers** because it uses the **median and IQR** instead of the mean and standard deviation.


## 1. What is Scaling?

Suppose we have two features:

- Age → `20–60`
- Salary → `20,000–200,000`

The features have very different ranges. Feature scaling changes values to a more comparable scale.

Common Scikit-Learn scalers:
- `StandardScaler`
- `MinMaxScaler`
- `RobustScaler`


## 2. What is an Outlier?

An **outlier** is a value that is unusually far from most other values.

Example:

```text
20, 21, 22, 23, 24, 25, 26, 27, 200
```

Here `200` is an outlier.

Outliers can strongly affect the **mean** and **standard deviation**. RobustScaler is designed to be less affected by extreme values.


## 3. Median and IQR

RobustScaler uses two important ideas.

### Median
The middle value after sorting.

```text
10, 20, 30, 40, 50
```

Median = `30`

### IQR
IQR means **Interquartile Range**.

```text
IQR = Q3 - Q1
```

- Q1 = 25th percentile
- Q3 = 75th percentile


## 4. RobustScaler Formula

The basic idea is:

$$
x_{scaled} = \frac{x - median}{IQR}
$$

So RobustScaler uses the **median and IQR**, which makes it less sensitive to extreme values.


In [ ]:
import numpy as np
from sklearn.preprocessing import RobustScaler

# Example data with an outlier
X = np.array([
    [10],
    [11],
    [12],
    [13],
    [14],
    [100]
])

print(X)


## 5. Create a RobustScaler

```python
scaler = RobustScaler()
```

At this point, the scaler has not learned anything yet. It is simply a scaler object.


In [ ]:
scaler = RobustScaler()
print(scaler)


## 6. `fit()`

`fit()` means:

> **Learn the required information from the data.**

For RobustScaler, it learns the median and scaling information based on the IQR.

```python
scaler.fit(X)
```

It does not return the scaled data.


In [ ]:
scaler.fit(X)

print("Center (median):", scaler.center_)
print("Scale (IQR):", scaler.scale_)


### Understanding `center_` and `scale_`

After `fit()`:

- `center_` stores the median for each feature.
- `scale_` stores the value used for scaling, based on the IQR.

The `_` means these values were learned during `fit()`.


## 7. `transform()`

After the scaler has learned the required values:

```python
X_scaled = scaler.transform(X)
```

`transform()` applies the learned scaling and returns a new scaled array.


In [ ]:
X_scaled = scaler.transform(X)

print("Original data:")
print(X)

print("\nScaled data:")
print(np.round(X_scaled, 2))


## 8. `fit_transform()`

Instead of:

```python
scaler.fit(X)
X_scaled = scaler.transform(X)
```

we can write:

```python
X_scaled = scaler.fit_transform(X)
```

It performs **fit + transform** together.


In [ ]:
scaler = RobustScaler()

X_scaled = scaler.fit_transform(X)

print(np.round(X_scaled, 2))


## 9. Why RobustScaler Helps With Outliers

Our data contains:

```text
10, 11, 12, 13, 14, 100
```

The value `100` is much larger than the others.

RobustScaler uses the **median and IQR**, which are less sensitive to extreme values than the mean and standard deviation.

> RobustScaler does **not remove** the outlier. It only scales the data in a way that is less affected by it.


## 10. Multiple Features

RobustScaler works **column by column**.

For example:

```text
Age        → its own median and IQR
Salary     → its own median and IQR
Experience → its own median and IQR
```


In [ ]:
X = np.array([
    [20, 25000, 1],
    [22, 30000, 2],
    [25, 40000, 3],
    [27, 45000, 4],
    [30, 50000, 5],
    [35, 300000, 10]  # salary outlier
])

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

print("Original data:")
print(X)

print("\nScaled data:")
print(np.round(X_scaled, 2))


## 11. Train-Test Data

Suppose we have 100 rows:

```text
100 rows
├── 80 rows → Training
└── 20 rows → Testing
```

The important rule is:

> **Fit the scaler only on training data.**

Then use the same scaler to transform both training and testing data.

This prevents **data leakage**.


In [ ]:
from sklearn.model_selection import train_test_split

X = np.array([
    [10], [12], [14], [16], [18],
    [20], [22], [24], [26], [28]
])

X_train, X_test = train_test_split(
    X,
    test_size=0.2,
    random_state=42
)

print("Training data:")
print(X_train)

print("\nTesting data:")
print(X_test)


## 12. Correct Scaling of Train and Test Data

The correct process is:

```python
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

### Remember

```text
Training data → fit + transform
Testing data  → transform only
```

The scaler learns its values from training data only.


In [ ]:
scaler = RobustScaler()

# Learn median and IQR only from training data
X_train_scaled = scaler.fit_transform(X_train)

# Use the same learned values for test data
X_test_scaled = scaler.transform(X_test)

print("Scaled training data:")
print(np.round(X_train_scaled, 2))

print("\nScaled testing data:")
print(np.round(X_test_scaled, 2))


## 13. What is Data Leakage?

Data leakage happens when information from test data accidentally influences the training process.

### Wrong

```python
X_scaled = scaler.fit_transform(X)
X_train, X_test = train_test_split(X_scaled)
```

The scaler learned information from the complete dataset, including test data.

### Correct

```python
X_train, X_test = train_test_split(X)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
```


## 14. RobustScaler vs StandardScaler vs MinMaxScaler

### StandardScaler
Uses **mean + standard deviation**.

$$
z = \frac{x - mean}{standard\ deviation}
$$

It can be affected by outliers.

### MinMaxScaler
Uses **minimum + maximum** and normally scales to `0–1`.

$$
x_{scaled} = \frac{x-x_{min}}{x_{max}-x_{min}}
$$

It is sensitive to extreme minimum or maximum values.

### RobustScaler
Uses **median + IQR**.

$$
x_{scaled} = \frac{x-median}{IQR}
$$

It is more resistant to outliers.

### Easy memory trick

```text
StandardScaler → Mean + Standard Deviation
MinMaxScaler   → Minimum + Maximum
RobustScaler   → Median + IQR
```


## 15. When Should I Use RobustScaler?

RobustScaler can be useful when:

- numerical features contain noticeable outliers
- you do not want extreme values to strongly affect scaling
- your machine learning algorithm benefits from feature scaling

There is no single scaler that is always best. The choice depends on the dataset and model.
